# 模块 D 综合项目：中文新闻分类与概率采样

模拟中文科技/体育新闻，用 jieba 分词、TF-IDF 和逻辑回归训练文本分类；理解预测概率形状并完成采样和评估。

**项目类型：**教师/自学参考答案 · 分步骤完整代码

## 提交要求

1. 准备中文新闻文本和标签，分层划分
2. 使用 jieba + TF-IDF + LogisticRegression Pipeline
3. 计算准确率、F1 和预测概率
4. 修正 np.random.choice 的 p 概率形状并执行采样
5. 导出预测 CSV、指标 JSON 和混淆矩阵 PNG

## 0. 准备工作：执行以下代码生成本地素材

这个单元格可以直接运行。素材为合成测试数据，无需联网。

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import jieba
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,f1_score,confusion_matrix

out=Path("D_results");out.mkdir(exist_ok=True)
sports=["足球 球队 比赛 进球","篮球 比赛 球员 得分","体育 比赛 运动员 训练",
        "球队 进球 赢得 赛事","运动 比赛 教练 球队"]
tech=["科技 软件 计算机 系统","人工智能 模型 机器学习","算法 训练 计算机 软件",
      "科技 模型 深度学习","人工智能 算法 应用 软件"]
texts=(sports+tech)*5
labels=np.array(([0]*5+[1]*5)*5)
print("文档:",len(texts),"标签分布:",np.bincount(labels))

## TODO 1：使用 jieba + TF-IDF + LogisticRegression Pipeline

运行参考实现，观察变量变化与输出。 

In [ ]:
Xtr,Xte,ytr,yte=train_test_split(texts,labels,test_size=0.2,random_state=42,stratify=labels)
pipe=Pipeline([
    ("tfidf",TfidfVectorizer(tokenizer=lambda s:list(jieba.cut(s)),token_pattern=None)),
    ("model",LogisticRegression(max_iter=300))
])
pipe.fit(Xtr,ytr)
pred=pipe.predict(Xte)
probs=pipe.predict_proba(Xte)
print("预测概率形状:",probs.shape)

## TODO 2：计算准确率、F1 和预测概率

运行参考实现，观察变量变化与输出。 

In [ ]:
acc=accuracy_score(yte,pred)
f1=f1_score(yte,pred,zero_division=0)
# np.random.choice 的 p 必须是一维，不可以直接传 (1,2) 或 (N,2)
sample_prob=probs[0]
assert sample_prob.ndim==1 and np.isclose(sample_prob.sum(),1)
sample=np.random.choice(pipe.classes_,p=sample_prob)
print("accuracy:",acc,"F1:",f1,"概率采样:",sample)

## TODO 3：修正 np.random.choice 的 p 概率形状并执行采样

运行参考实现，观察变量变化与输出。 

In [ ]:
pd.DataFrame({"text":Xte,"y_true":yte,"y_pred":pred}).to_csv(
    out/"D_predictions.csv",index=False,encoding="utf-8-sig")
(out/"D_metrics.json").write_text(json.dumps({"accuracy":float(acc),"f1":float(f1)},ensure_ascii=False,indent=2),encoding="utf-8")
cm=confusion_matrix(yte,pred,labels=[0,1])
fig,ax=plt.subplots();ax.imshow(cm,cmap="Blues")
for i in range(2):
    for j in range(2):ax.text(j,i,str(cm[i,j]),ha="center",va="center")
ax.set_title("Chinese news classification");ax.set_xlabel("Predicted");ax.set_ylabel("Actual")
fig.tight_layout();fig.savefig(out/"D_confusion.png",dpi=150);plt.close(fig)

## TODO 4：导出预测 CSV、指标 JSON 和混淆矩阵 PNG

运行参考实现，观察变量变化与输出。 

In [ ]:
for name in ["D_predictions.csv","D_metrics.json","D_confusion.png"]:
    assert (out/name).is_file() and (out/name).stat().st_size>0,name
assert len(pd.read_csv(out/"D_predictions.csv"))==len(yte)
print("D 模块全部交付文件核验通过")

## 交付核验

- `D_results/D_predictions.csv`
- `D_results/D_metrics.json`
- `D_results/D_confusion.png`

完成后核对所有文件是否存在并可打开。参考答案在 `solutions/D_project_solution.ipynb`。

In [ ]:
# 交付文件自检：仅检查文件是否存在且非空，不自动判断模型效果
from pathlib import Path
folder=Path("D_results")
required=["D_predictions.csv","D_metrics.json","D_confusion.png"]
missing=[name for name in required if not (folder/name).is_file() or (folder/name).stat().st_size==0]
if missing:
    print("缺失或为空的文件：",missing)
else:
    print("全部交付文件已生成")
